# LoRA: Low-Rank Adaptation - 실습 코드 1: LoRA 구현 및 적용 (PEFT)

- Tutorial ID: `expand-lora`
- Tutorial: LoRA: Low-Rank Adaptation
- Section ID: `expand-lora-code-1`
- Section: 실습 코드 1: LoRA 구현 및 적용 (PEFT)


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: LoRA 구현 및 적용 (PEFT)
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) 큰 가중치 W를 직접 바꾸지 않고 저랭크 A/B 업데이트가 어떻게 더해지는지 확인
#   3) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

# 기본 모델 로드
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    load_in_4bit=True,  # QLoRA: 4-bit 양자화
    device_map="auto"
)

# LoRA 설정
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
        r=8,                    # Low rank
    lora_alpha=16,          # Scaling factor
        lora_dropout=0.05,
    target_modules=[        # 적용 대상
        "q_proj", "k_proj",
        "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

# LoRA 모델 생성
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# 출력: trainable params: 13,107,200 || all params: 6,738,415,616 || 0.19%

# 학습 후 LoRA 가중치만 저장 (수 MB!)
model.save_pretrained("./lora-weights")